# 🎯 Hallucination Hunter - GRPO Training

**Training Method**: GRPO (Group Relative Policy Optimization) - RL Training

**Model**: Qwen2.5-3B-Instruct with Unsloth

**Environment**: https://huggingface.co/spaces/tusharpawar21/hallicunation-Hunt

In [ ]:
# Install dependencies
!pip install -q unsloth trl transformers accelerate bitsandbytes httpx matplotlib

In [ ]:
import torch
import httpx
import json
import re
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainingArguments
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Model with Unsloth

In [ ]:
# Load Qwen2.5-3B with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("✅ Model loaded with LoRA")

## 2. Environment Wrapper for GRPO

In [ ]:
class HallucinationEnvGRPO:
    """Environment wrapper for GRPO training"""
    
    def __init__(self, base_url):
        self.base_url = base_url.rstrip('/')
        self.client = httpx.Client(timeout=60.0)
        self.current_prompt = None
        
    def get_prompt(self):
        """Get a new prompt from environment"""
        response = self.client.post(f"{self.base_url}/reset")
        response.raise_for_status()
        data = response.json()
        obs = data['observation']
        
        # Format as prompt for model
        prompt = f"""Analyze this text for hallucinations. For each claim, determine if it's factual, hallucinated, or unverifiable.

Text: {obs['generated_text']}

Output JSON format:
{{
  "detected_claims": [
    {{
      "claim_text": "exact claim from text",
      "label": "factual" or "hallucinated" or "unverifiable",
      "reason": "explanation",
      "corrected_fact": "correction if hallucinated, else null"
    }}
  ]
}}

Analysis:"""
        self.current_prompt = prompt
        return prompt
    
    def compute_reward(self, response_text):
        """Compute reward for model response"""
        try:
            # Extract JSON from response
            json_match = re.search(r'\{[^{}]*"detected_claims"[^{}]*\[[^\]]*\][^{}]*\}', response_text, re.DOTALL)
            if not json_match:
                return -5.0  # Penalty for invalid format
            
            json_str = json_match.group(0)
            detection = json.loads(json_str)
            
            # Send to environment
            action = {"detection_output": detection}
            response = self.client.post(f"{self.base_url}/step", json=action)
            response.raise_for_status()
            result = response.json()
            
            return result['reward']
        except json.JSONDecodeError:
            return -5.0  # Penalty for invalid JSON
        except Exception as e:
            print(f"Error computing reward: {e}")
            return -3.0  # Penalty for other errors

# Initialize environment
ENV_URL = "https://tusharpawar21-hallicunation-hunt.hf.space"
env = HallucinationEnvGRPO(ENV_URL)

# Test connection
try:
    health = env.client.get(f"{ENV_URL}/health").json()
    print(f"✅ Connected to environment")
    print(f"Episodes: {health['episode_count']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

## 3. GRPO Training Configuration

In [ ]:
# GRPO Configuration
config = GRPOConfig(
    output_dir="./hallucination-hunter-grpo",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    logging_steps=5,
    save_steps=50,
    max_steps=200,
    temperature=0.8,
    fp16=True,
)

print("✅ GRPO config created")

## 4. Prepare Training Dataset

In [ ]:
# Prepare dataset for GRPO
from datasets import Dataset

# Collect prompts from environment
print("Collecting training prompts...")
prompts = []
for i in range(50):  # Collect 50 prompts
    prompt = env.get_prompt()
    prompts.append({"query": prompt})
    if (i + 1) % 10 == 0:
        print(f"  Collected {i+1}/50 prompts")

# Create dataset
train_dataset = Dataset.from_list(prompts)
print(f"\n✅ Dataset created with {len(train_dataset)} prompts")

# Define reward function
def reward_function(samples):
    """Compute rewards for GRPO training"""
    rewards = []
    for sample in samples:
        # Extract response after the prompt
        response = sample.split("Analysis:")[-1].strip() if "Analysis:" in sample else sample
        reward = env.compute_reward(response)
        rewards.append(reward)
    return rewards

print("✅ Reward function created")

## 5. Create GRPO Trainer

In [ ]:
# Create trainer with dataset
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=400,
    temperature=0.8,
    do_sample=True,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=config,
    train_dataset=train_dataset,
    reward_model=reward_function,
    generation_config=generation_config,
)

print("✅ GRPO Trainer created")

## 6. Train with GRPO

In [ ]:
# Collect baseline rewards
print("Collecting baseline rewards...")
baseline_rewards = []
for i in range(10):
    prompt = env.get_prompt()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    reward = env.compute_reward(response[len(prompt):])
    baseline_rewards.append(reward)
    print(f"Baseline {i+1}/10: {reward:.3f}")

print(f"\nBaseline avg: {np.mean(baseline_rewards):.3f}")

# Train
print("\n🚀 Starting GRPO training...\n")
trainer.train()
print("\n✅ Training complete!")

## 7. Evaluate Trained Model

In [ ]:
# Collect trained rewards
print("Evaluating trained model...")
FastLanguageModel.for_inference(model)
trained_rewards = []

for i in range(10):
    prompt = env.get_prompt()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    reward = env.compute_reward(response[len(prompt):])
    trained_rewards.append(reward)
    print(f"Trained {i+1}/10: {reward:.3f}")

print(f"\nTrained avg: {np.mean(trained_rewards):.3f}")
print(f"Improvement: {np.mean(trained_rewards) - np.mean(baseline_rewards):+.3f}")

## 8. Visualize Results

In [ ]:
# Extract training history
history = trainer.state.log_history
steps = []
rewards = []
losses = []

for entry in history:
    if 'reward' in entry:
        steps.append(entry.get('step', 0))
        rewards.append(entry['reward'])
    if 'loss' in entry:
        losses.append(entry['loss'])

# Create plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Reward curve
if rewards:
    axes[0].plot(steps, rewards, 'b-', linewidth=2, marker='o', markersize=4)
    axes[0].set_xlabel('Training Steps')
    axes[0].set_ylabel('Reward')
    axes[0].set_title('Reward Over Time (GRPO)')
    axes[0].grid(True, alpha=0.3)

# Loss curve
if losses:
    axes[1].plot(range(len(losses)), losses, 'r-', linewidth=2)
    axes[1].set_xlabel('Training Steps')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Loss Over Time')
    axes[1].grid(True, alpha=0.3)

# Before/After comparison
comparison = {
    'Before\nTraining': np.mean(baseline_rewards),
    'After\nTraining': np.mean(trained_rewards)
}
colors = ['#ff6b6b', '#51cf66']
bars = axes[2].bar(comparison.keys(), comparison.values(), color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[2].set_ylabel('Average Reward')
axes[2].set_title('Before vs After Training')
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].axhline(y=0, color='black', linestyle='--', linewidth=1)

for i, (k, v) in enumerate(comparison.items()):
    axes[2].text(i, v + 0.2, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('grpo_training_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Plots saved to grpo_training_results.png")

## 9. Save Model

In [ ]:
# Save LoRA adapters
model.save_pretrained("hallucination-hunter-grpo-lora")
tokenizer.save_pretrained("hallucination-hunter-grpo-lora")

print("✅ Model saved")

## 10. Summary

In [ ]:
print("\n" + "="*80)
print("GRPO TRAINING SUMMARY")
print("="*80)
print(f"\nModel: Qwen2.5-3B-Instruct (4-bit + LoRA r=16)")
print(f"Training Method: GRPO (Reinforcement Learning)")
print(f"Total Steps: {len(steps)}")

if losses:
    print(f"\nLoss: {losses[0]:.4f} → {losses[-1]:.4f}")
    print(f"Improvement: {((losses[0] - losses[-1]) / losses[0] * 100):.1f}%")

print(f"\nRewards:")
print(f"  Before: {np.mean(baseline_rewards):.3f} (±{np.std(baseline_rewards):.3f})")
print(f"  After: {np.mean(trained_rewards):.3f} (±{np.std(trained_rewards):.3f})")
print(f"  Improvement: {np.mean(trained_rewards) - np.mean(baseline_rewards):+.3f}")

print(f"\n✅ GRPO training complete!")